In [1]:
# Data Cleaning Notebook (Data PreProcessing)
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# --- 1. Load Data ---
# We use header=[0,1] because yfinance CSVs often have two header rows
cse = pd.read_csv('../data/raw/cse_historical_data.csv', header=[0, 1], index_col=0)
macro = pd.read_csv('../data/raw/srilanka_macro_data.csv')

# Flatten Multi-index columns if they exist
cse.columns = cse.columns.get_level_values(0)
cse.index = pd.to_datetime(cse.index)
cse = cse.reset_index()

# --- 2. Reshape Macro Data ---
# Convert from wide (years as columns) to long format
macro_long = macro.melt(id_vars=['series'], var_name='Year', value_name='Value')
macro_long['Year'] = macro_long['Year'].str.replace('YR', '').astype(int)

# Pivot so Inflation and GDP_Growth are separate columns
macro_pivot = macro_long.pivot(index='Year', columns='series', values='Value').reset_index()
macro_pivot.columns = ['Year', 'Inflation', 'GDP_Growth']

# --- 3. Feature Engineering (Date Handling) ---
cse['Year'] = cse['Date'].dt.year
cse['Month'] = cse['Date'].dt.month
cse['Day'] = cse['Date'].dt.day

# --- 4. Merging ---
# Merge daily stock data with yearly macro data on 'Year'
df = pd.merge(cse, macro_pivot, on='Year', how='left')

# --- 5. Normalization ---
# We normalize features to a range of [0, 1] using the formula:
# $x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$

scaler = MinMaxScaler()
feature_cols = ['Inflation', 'GDP_Growth', 'Month', 'Day']
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# --- 6. Final Clean up & Save ---
# Drop rows with NaN values (if any)
df = df.dropna()

# Save to the PROCESSED data folder
os.makedirs('../data/processed', exist_ok=True)
df.to_csv("../data/processed/processed_data.csv", index=False)

print("Data cleaning complete. Processed file saved in data/processed/processed_data.csv")
print(df.head())

Data cleaning complete. Processed file saved in data/processed/processed_data.csv
        Date        Close         High          Low         Open     Volume  \
0 2018-01-02  6411.270020  6431.160156  6367.950195  6368.049805    5047700   
1 2018-01-03  6463.500000  6465.620117  6411.270020  6420.720215   12970500   
2 2018-01-04  6459.660156  6469.379883  6430.629883  6465.370117      30300   
3 2018-01-05  6514.729980  6520.799805  6459.660156  6462.850098   19629500   
4 2018-01-08  6540.509766  6545.160156  6513.549805  6513.549805  108728700   

   Year  Month       Day  Inflation  GDP_Growth  
0  2018    0.0  0.033333        0.0    0.785168  
1  2018    0.0  0.066667        0.0    0.785168  
2  2018    0.0  0.100000        0.0    0.785168  
3  2018    0.0  0.133333        0.0    0.785168  
4  2018    0.0  0.233333        0.0    0.785168  
